# Deception Classifier — Colab GPU Training
Run each cell in order. Make sure the runtime is set to **T4 GPU**:  
Runtime → Change runtime type → T4 GPU

## 1. Clone repo & install dependencies

In [1]:
!git clone https://github.com/Lucca878/deceptionClassifierTraining.git
%cd deceptionClassifierTraining

Cloning into 'deceptionClassifierTraining'...
remote: Enumerating objects: 153, done.
remote: Counting objects: 100% (153/153), done.
remote: Compressing objects: 100% (145/145), done.
remote: Total 153 (delta 2), reused 153 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (153/153), 4.29 MiB | 17.17 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/deceptionClassifierTraining


In [2]:
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 113.6 MB/s eta 0:00:0000:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 89.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 7.9 MB/s eta 0:00:00


## 2. Verify GPU

In [3]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

GPU available: True
Device: NVIDIA A100-SXM4-40GB


## 3. Train DistilBERT

In [4]:
!python src/pipeline/run_pipeline.py \
  --mode train \
  --model distilbert \
  --output_root ./outputs \
  --seed 42

Training model preset: distilbert
Backbone: distilbert-base-uncased
Traceback (most recent call last):
  File "/content/deceptionClassifierTraining/src/pipeline/run_pipeline.py", line 73, in <module>
    main()
  File "/content/deceptionClassifierTraining/src/pipeline/run_pipeline.py", line 57, in main
    trained_model_dir = run_training(cfg)
                        ^^^^^^^^^^^^^^^^^
  File "/content/deceptionClassifierTraining/src/pipeline/train.py", line 290, in run_training
    cv_results = run_cv_training(df, cfg, run_dir)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/deceptionClassifierTraining/src/pipeline/train.py", line 150, in run_cv_training
    accuracy_metric = evaluate.load("accuracy")
                      ^^^^^^^^^^^^^
AttributeError: module 'evaluate' has no attribute 'load'


## 4. Check output

In [ ]:
import os, glob

# Find the latest training run
run_dirs = sorted(glob.glob("outputs/distilbert_*"))
latest = run_dirs[-1] if run_dirs else None
print("Latest run:", latest)

if latest:
    for f in os.listdir(latest):
        print(" ", f)

In [ ]:
import pandas as pd

cv_path = os.path.join(latest, "cv_results.csv") if latest else None
if cv_path and os.path.exists(cv_path):
    df = pd.read_csv(cv_path)
    print(df)
    print("\nMean accuracy:", df["eval_accuracy"].mean().round(4))
    print("Mean F1:      ", df["eval_f1"].mean().round(4))

## 5. Download the trained model

In [ ]:
import shutil

model_dir = os.path.join(latest, "model") if latest else None
zip_path = "/content/distilbert_retrained"

if model_dir and os.path.exists(model_dir):
    shutil.make_archive(zip_path, "zip", model_dir)
    print("Zipped:", zip_path + ".zip")
else:
    print("Model dir not found:", model_dir)

In [ ]:
from google.colab import files
files.download("/content/distilbert_retrained.zip")